# 03 — Parsing MODS into a hearing table

**Purpose.** This notebook turns 36,378 MODS records into one table with one row
per hearing, carrying the committee that held it, the members listed as present,
the witnesses, the sitting dates, and whether the transcript was ever digitized.
Everything the later stages select on is decided here, so this is where the
corpus stops being a directory of files and becomes a sample with known
properties.

The two facts this table exists to make visible are both ones the earlier
pipeline absorbed silently:

- **A hearing with no `congMember` cannot pass through topic extraction.** That
  stage attributes statements by matching member surnames against paragraphs,
  so an empty member list yields no attributable statements and the hearing
  drops out with no record of having done so. It affected 5.4 percent of the
  earlier corpus.
- **A hearing can download perfectly and contain no transcript.** GovInfo serves
  a cover page carrying the literal string `<TEXT NOT AVAILABLE>` for hearings
  it never digitized past the front matter — close to half of 1995–96. The
  earlier pipeline removed these with a `char_count > 8650` threshold whose
  justification was a comment saying the number had been checked by eye.

**Inputs.**

| Source | Path | Contents |
|---|---|---|
| Package manifest | `GOVINFO_MANIFEST / packages_all.parquet` | The work list, from `01`. |
| Fetch status | `GOVINFO_STATUS / fetch_summary.parquet` | What `02` actually wrote, and what it could not. |
| MODS records | `GOVINFO_RAW / {year} / mods / {packageId}.xml` | Parsed by `lib/mods.py`. |
| Transcripts | `GOVINFO_RAW / {year} / html / {packageId}.htm` | Read only for size and the availability marker. |

**Method.**

1. Take the work list from the fetch status rather than from a directory glob.
   Which files exist is the question, not the premise; globbing makes a missing
   file indistinguishable from a file that was never meant to be there.
2. Parse each MODS record with `lib/mods.py`. Committees, members, and witnesses
   are kept as lists on the row; the relational tables in step 3 are derived
   from them rather than parsed separately, so the two cannot disagree.
3. Keep **every** `congCommittee`, not just the first. Joint hearings list more
   than one, and the earlier pipeline's `.//congCommittee` took whichever came
   first without recording that a choice had been made. `n_committees` marks the
   rows where the choice matters.
4. Keep members whose `bioGuideId` is absent, with a null id, rather than
   dropping them. `n_members_with_bioguide` alongside `n_members` is what makes
   the unattributable ones countable.
5. Read the transcript's availability from the marker GovInfo puts in the file,
   not from a length threshold. The check is gated on file size so a 6 MB
   transcript is not scanned for a string that only appears in 2 KB stubs.
6. Write three parquet tables — hearings, hearing-committee pairs,
   hearing-member pairs — and report coverage by year.

**Outputs.**

| File | Path | Contents |
|---|---|---|
| Hearing table | `INTERIM / govinfo / hearings.parquet` | One row per package: identity, dates, committee, counts, transcript availability. |
| Committee pairs | `INTERIM / govinfo / hearing_committees.parquet` | One row per hearing-committee pair; joint hearings appear more than once. |
| Member pairs | `INTERIM / govinfo / hearing_members.parquet` | One row per hearing-member pair. |
| Coverage table | `INTERIM / govinfo / coverage_by_year.csv` | Per year: hearings, with committee, with members, with transcript text. |

**Notes.**

The member list in MODS is the committee's roster as catalogued, not the set of
people who actually spoke at that sitting. It is an upper bound on attributable
speakers, and the surname matching in the topic-extraction stage narrows it. A
consequence worth stating: `congress` on a `congMember` element sometimes
disagrees with `congress` on the package — a 105th Congress hearing carrying a
roster tagged 106th — because the roster was catalogued when the package was
ingested rather than when the hearing was held. The hearing's own `congress` is
authoritative here, and the member-level value is carried through unchanged so
the discrepancy stays visible.

## Setup

`lib/mods.py` holds the parsing; this notebook orchestrates and reports. The
only constant declared here is the output directory, because the work list, the
year partition, and the file paths all come from the tables `01` and `02` wrote.

In [1]:
import sys

sys.path.insert(0, "../lib")

import pandas as pd

from mods import parse_mods, transcript_status
from paths import GOVINFO_MANIFEST, GOVINFO_RAW, GOVINFO_STATUS, INTERIM

OUT = INTERIM / "govinfo"
OUT.mkdir(parents=True, exist_ok=True)

manifest = pd.read_parquet(GOVINFO_MANIFEST / "packages_all.parquet")
status = pd.read_parquet(GOVINFO_STATUS / "fetch_summary.parquet")

print(f"manifest: {len(manifest):,} packages")
print(f"status  : {len(status):,} fetch outcomes")
print(status["status"].value_counts().to_string())

manifest: 36,378 packages
status  : 73,221 fetch outcomes
status
ok         72346
missing      875


## Step 1 — Parse every MODS record

The work list is the set of packages whose MODS landed, taken from the status
log. A package whose MODS is absent cannot be parsed, and the count of those is
printed rather than allowed to shrink the corpus quietly — `02`'s verification
cell already refuses to pass while any exist, so this should read zero.

In [2]:
mods_ok = status.loc[(status["kind"] == "mods") & status["status"].isin(["ok", "skipped"])]
missing_mods = set(manifest["packageId"]) - set(mods_ok["packageId"])
print(f"{len(mods_ok):,} MODS records to parse, {len(missing_mods):,} packages without one")

records = []
failures = []
for path in mods_ok["path"]:
    try:
        records.append(parse_mods(path))
    except Exception as exc:  # a malformed record is data, not a reason to stop
        failures.append({"path": path, "error": f"{type(exc).__name__}: {exc}"})

hearings = pd.DataFrame.from_records(records)
print(f"parsed {len(hearings):,} records, {len(failures)} failed")
if failures:
    print(pd.DataFrame(failures).head(10).to_string(index=False))

36,378 MODS records to parse, 0 packages without one


parsed 36,378 records, 0 failed


## Step 2 — Attach the transcript's availability

`transcript_status` reports three things per package: whether the HTML file is
on disk, how large it is, and whether it contains a transcript rather than a
cover page. The third is the one that matters and the one a file listing cannot
answer.

In [3]:
year_of = dict(zip(manifest["packageId"], manifest["query_year"]))
hearings["year"] = hearings["packageId"].map(year_of)

html_paths = [
    GOVINFO_RAW / str(year) / "html" / f"{package_id}.htm"
    for package_id, year in zip(hearings["packageId"], hearings["year"])
]

text_status = pd.DataFrame.from_records([transcript_status(p) for p in html_paths])
hearings["html_bytes"] = text_status["bytes"].to_numpy()
hearings["has_transcript"] = text_status["has_text"].to_numpy()
hearings["html_on_disk"] = text_status["exists"].to_numpy()

print(f"{hearings['html_on_disk'].sum():,} transcripts on disk")
print(f"{hearings['has_transcript'].sum():,} contain text "
      f"({1 - hearings['has_transcript'].mean():.1%} are cover-page stubs)")

35,503 transcripts on disk
35,125 contain text (3.4% are cover-page stubs)


## Step 3 — Explode committees and members into their own tables

The nested list columns are convenient on the hearing row and useless for a
join. These two tables are derived from those same lists, so a committee that
appears in one appears in the other by construction.

In [4]:
committee_rows = [
    {"packageId": row.packageId, "position": i, **committee}
    for row in hearings.itertuples()
    for i, committee in enumerate(row.committees)
]
member_rows = [
    {"packageId": row.packageId, **member}
    for row in hearings.itertuples()
    for member in row.members
]

hearing_committees = pd.DataFrame.from_records(committee_rows)
hearing_members = pd.DataFrame.from_records(member_rows)

print(f"hearing-committee pairs: {len(hearing_committees):,} "
      f"across {hearing_committees['packageId'].nunique():,} hearings")
print(f"hearing-member pairs   : {len(hearing_members):,} "
      f"across {hearing_members['packageId'].nunique():,} hearings")
print(f"distinct committees    : {hearing_committees['name_standard'].nunique():,}")
print(f"distinct bioGuideIds   : {hearing_members['bioGuideId'].nunique():,}")

hearing-committee pairs: 37,258 across 36,366 hearings
hearing-member pairs   : 1,031,089 across 31,930 hearings
distinct committees    : 83
distinct bioGuideIds   : 1,554


## Step 4 — Coverage by year

Three columns decide whether a hearing can be used by the later stages: it needs
a committee to be selected on, members to attribute statements to, and a
transcript to read. This table is the honest denominator for every count the
pipeline reports downstream, and it is the figure a reader should look at before
believing any trend over time — a series that rises because coverage rises is
not a finding.

In [5]:
coverage = (
    hearings.assign(
        has_committee=hearings["n_committees"] > 0,
        has_members=hearings["n_members"] > 0,
        has_bioguide=hearings["n_members_with_bioguide"] > 0,
    )
    .groupby("year")
    .agg(
        hearings=("packageId", "size"),
        with_committee=("has_committee", "sum"),
        with_members=("has_members", "sum"),
        with_bioguide=("has_bioguide", "sum"),
        with_transcript=("has_transcript", "sum"),
    )
)
coverage["usable"] = (
    hearings.assign(
        u=lambda d: (d["n_committees"] > 0) & (d["n_members"] > 0) & d["has_transcript"]
    )
    .groupby("year")["u"].sum()
)
for column in ("with_committee", "with_members", "with_transcript", "usable"):
    coverage[column.replace("with_", "") + "_pct"] = (
        coverage[column] / coverage["hearings"]
    ).map("{:.0%}".format)

print(coverage[["hearings", "with_committee", "with_members", "with_transcript",
                "usable", "usable_pct"]].to_string())

      hearings  with_committee  with_members  with_transcript  usable usable_pct
year                                                                            
1995       253             253             2              147       2         1%
1996       141             141             2               77       2         1%
1997       456             456           394              455     394        86%
1998       400             400           279              374     279        70%
1999       850             850           774              823     774        91%
2000       837             835           760              805     758        91%
2001      1125            1125           999             1056     999        89%
2002      1143            1142           998             1067     998        87%
2003      1365            1364          1190             1273    1190        87%
2004      1101            1101           972             1062     972        88%
2005      1521            15

## Step 5 — Persist

Parquet, partitioned nowhere: these tables are a few tens of MB and every later
stage reads all of them. The list columns survive the round trip as arrays,
which is why the hearing table can stay the single source and the two pair
tables stay derived.

In [6]:
hearings.to_parquet(OUT / "hearings.parquet", index=False)
hearing_committees.to_parquet(OUT / "hearing_committees.parquet", index=False)
hearing_members.to_parquet(OUT / "hearing_members.parquet", index=False)
coverage.to_csv(OUT / "coverage_by_year.csv")

for name in ("hearings", "hearing_committees", "hearing_members"):
    path = OUT / f"{name}.parquet"
    print(f"{path.name:28s} {path.stat().st_size / 1e6:>7.1f} MB")

hearings.parquet                12.4 MB
hearing_committees.parquet       0.4 MB
hearing_members.parquet          3.6 MB


## Step 6 — The committees the demonstration subset is drawn from

The House Science Committee was renamed twice, so selecting it by current name
silently drops two thirds of its history. Printing the three names with their
year ranges is what makes the lineage a stated choice rather than a filter
someone has to reverse-engineer later.

The count printed here is also a check on the collection: an earlier run of this
pipeline held 870 hearings in this lineage against 891 in a re-pull, and the
difference was almost entirely hearings published after that pull. A number far
from either would mean the enumeration or the committee parse is wrong.

In [7]:
LINEAGE = [
    "Committee on Science",
    "Committee on Science and Technology",
    "Committee on Science, Space, and Technology",
]

# GovInfo's committee strings carry stray whitespace — a doubled space after
# "Committee on", a trailing space — so an exact match drops real rows. Normalize
# before comparing, and report how many rows the normalization rescued.
normalized = hearing_committees["name_standard"].str.replace(r"\s+", " ", regex=True).str.strip()
exact = hearing_committees["name_standard"].isin(LINEAGE)
loose = normalized.isin(LINEAGE)
print(f"whitespace normalization rescued {int((loose & ~exact).sum())} committee rows")

lineage = hearing_committees.loc[loose, ["packageId", "name_standard", "authorityId"]]
lineage = lineage.assign(committee=normalized[loose]).merge(
    hearings[["packageId", "year", "has_transcript", "n_members"]], on="packageId"
)

print()
print(lineage.groupby("committee").agg(
    hearings=("packageId", "nunique"),
    first_year=("year", "min"),
    last_year=("year", "max"),
    with_transcript=("has_transcript", "sum"),
).to_string())
print(f"\nlineage total: {lineage['packageId'].nunique():,} hearings")
print()
print(lineage.groupby("year")["packageId"].nunique().to_string())

whitespace normalization rescued 0 committee rows

                                             hearings  first_year  last_year  with_transcript
committee                                                                                    
Committee on Science                              147        1996       2006              146
Committee on Science and Technology               231        1998       2010              224
Committee on Science, Space, and Technology       586        2011       2025              589

lineage total: 964 hearings

year
1996     1
1997     3
1998     4
1999    17
2000     2
2001     1
2003    37
2004    31
2005    34
2006    31
2007    62
2008    47
2009    66
2010    42
2011    56
2012    54
2013    56
2014    40
2015    55
2016    41
2017    44
2018    31
2019    61
2020    22
2021    41
2022    32
2023    29
2024    17
2025     7


## Verification

Five checks, each naming the failure it catches.

1. **No MODS record failed to parse.** A malformed record is collected into
   `failures` rather than raised so one bad file cannot end a 36,000-record run,
   but a non-zero count means the parser and the data disagree and the table is
   short by that many rows.
2. **One row per package, and every package accounted for.** A duplicate would
   mean two files claimed the same `packageId`; a shortfall means a parsed
   record was dropped between step 1 and the table.
3. **The pair tables reconcile with the counts on the hearing row.** `n_members`
   is computed during parsing and the member table is exploded from the same
   list, so a mismatch means the two views of the same data have diverged —
   exactly the failure that makes a nested column dangerous.
4. **`has_transcript` is never true for a package with no file on disk.**
   Guards the availability flag against being read off a file that is not there.
5. **Committee and member coverage are reported, not asserted.** Both are
   properties of GovInfo's cataloguing, so the check prints the rates and the
   worst years. What would be a defect is not knowing them.

In [8]:
assert not failures, f"{len(failures)} MODS records failed to parse; first: {failures[0]}"
print(f"1) all {len(hearings):,} MODS records parsed")

dupes = int(hearings["packageId"].duplicated().sum())
assert dupes == 0, f"{dupes} duplicate packageId in the hearing table"
assert len(hearings) == len(mods_ok), (
    f"{len(mods_ok):,} MODS files but {len(hearings):,} rows — records lost after parsing"
)
print(f"2) {len(hearings):,} rows, one per package, none lost")

from_pairs = hearing_members.groupby("packageId").size()
from_row = hearings.set_index("packageId")["n_members"]
diff = from_row.subtract(from_pairs, fill_value=0)
assert (diff == 0).all(), f"member counts disagree for {int((diff != 0).sum())} hearings"
print(f"3) member table reconciles with n_members on every row")

bad_flag = int((hearings["has_transcript"] & ~hearings["html_on_disk"]).sum())
assert bad_flag == 0, f"{bad_flag} hearings flagged as having text with no file on disk"
print("4) has_transcript implies the file is on disk")

no_committee = int((hearings["n_committees"] == 0).sum())
no_members = int((hearings["n_members"] == 0).sum())
no_text = int((~hearings["has_transcript"]).sum())
usable = int(((hearings["n_committees"] > 0) & (hearings["n_members"] > 0)
              & hearings["has_transcript"]).sum())
print(f"5) coverage over {len(hearings):,} hearings")
print(f"     no committee : {no_committee:>6,} ({no_committee / len(hearings):.1%})")
print(f"     no members   : {no_members:>6,} ({no_members / len(hearings):.1%})")
print(f"     no transcript: {no_text:>6,} ({no_text / len(hearings):.1%})")
print(f"     usable       : {usable:>6,} ({usable / len(hearings):.1%})")
print()
worst = coverage.nsmallest(5, "usable")[["hearings", "usable", "usable_pct"]]
print("   years with the least usable coverage:")
print(worst.to_string())

1) all 36,378 MODS records parsed
2) 36,378 rows, one per package, none lost
3) member table reconciles with n_members on every row
4) has_transcript implies the file is on disk
5) coverage over 36,378 hearings
     no committee :     12 (0.0%)
     no members   :  4,448 (12.2%)
     no transcript:  1,253 (3.4%)
     usable       : 31,916 (87.7%)

   years with the least usable coverage:
      hearings  usable usable_pct
year                             
1995       253       2         1%
1996       141       2         1%
1998       400     279        70%
1997       456     394        86%
2020       689     638        93%
